In [29]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

In [30]:
#อ่านไฟล์จาก test_set.csv

df = pd.read_csv('test_set.csv')
df.head(5)

,query log,status,label
0,2025-12-26T23:22:25.410030Z\t97\tQuery\tselect...,normal,1
1,2025-12-26T23:22:26.239664Z\t95\tQuery\tinsert...,normal,1
2,2025-12-26T23:22:26.714509Z\t38\tQuery\tselect...,anormaly,0
3,2025-12-26T23:22:27.003408Z\t49\tQuery\tselect...,normal,1
4,2025-12-26T23:22:27.230506Z\t58\tQuery\tselect...,normal,1


In [31]:
#โหลด Model
path = 'Finetuned Bert Model\checkpoint-60000'
tokenizer = AutoTokenizer.from_pretrained(path)
model = AutoModelForSequenceClassification.from_pretrained(path)

#บังคับว่าต้อง inference ที่ GPU (ถ้ามี)
model.to(device)
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
import re

def clean_log(text):
    # ลบ timestamp (optional แต่แนะนำ)
    text = re.sub(r"\d{4}-\d{2}-\d{2}T.*?Z", "", text)
    # แปลง tab เป็น space
    text = text.replace("\t", " ")
    return text.strip()

def predict_log(log_text):
    log_text = clean_log(log_text)
    inputs = tokenizer(
        log_text,
        return_tensors="pt",
        truncation=True,
        padding=True, # ใส่เผื่อเอาไว้ตอน inference มากกว่า 1 log (Batch Size > 1)
        max_length=128
    )
    # ⭐ ย้าย input ไป GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
        pred = torch.argmax(logits, dim=1).item()
        prob = torch.softmax(logits, dim=-1).tolist()[0]

    return "normal" if pred == 1 else "anormaly", prob

In [ ]:
#วัด Accuracy

correct_predictions = 0
total_predictions = len(df)

for index, row in df.iterrows():
    text_to_classify = row['query log'] # ใช้คอลัมน์ 'query log' เป็น input
    true_label = row['status'] # ใช้คอลัมน์ 'status' เป็นคำตอบจริง (normal หรือ anomaly)

    # ทำนาย
    prediction_result,confidence = predict_log(text_to_classify)

    # ตรวจสอบว่าทำนายถูกต้องหรือไม่
    if prediction_result == true_label:
        correct_predictions += 1
        correction = 'True'
    else:
        correction = 'False'
    print(f"prediction = {prediction_result} | true_status = {true_label} | correction = {correction} | confidence = {confidence}")

In [39]:
# วัด Accuracy
accuracy = (correct_predictions / total_predictions) * 100
print(f"test_set จำนวน {total_predictions}")
print(f"ทำนายถูกจำนวน {correct_predictions}")
print(f"Accuracy = {accuracy}")

test_set จำนวน 300000
ทำนายถูกจำนวน 300000
Accuracy = 100.0
